# Comparing Futures and ETF Volume

Unlike other notebooks in this course,
the program's Databento license cannot retrieve the data in
this notebook because it uses ETF data, which is not from the CME.
You should look at the outputs
in the notebook, but you need a different Databento API key to run it.

Following the examples in the Introduction to Futures notebook,
I show some volume comparisons
between ES futures and ETFs.

In [1]:
# Import and check venv
import datetime
import sys
from collections.abc import Iterable
from itertools import cycle
from pathlib import Path
from zoneinfo import ZoneInfo

import databento as db
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from finm37000 import (
    disk_cache,
    get_databento_api_key,
    get_official_stats,
    temp_env,
    us_business_day,
)

pio.templates.default = "plotly_dark"
px.defaults.color_discrete_sequence = px.colors.qualitative.Set3
color_palette = cycle(px.defaults.color_discrete_sequence)

sys.executable

'/Users/ericpatterson/Documents/github/finm37000-main/.venv/bin/python'

In [2]:
# Construct databento clients to retrieve data using different API keys.
# Note that the license for equity data is not included with the course;
# I am using a personal API key to retrieve this data.

with temp_env(
    DATABENTO_API_KEY=get_databento_api_key(Path("~/.databento/personal").expanduser())
):
    equity_client = db.Historical()


with temp_env(DATABENTO_API_KEY=get_databento_api_key()):
    futures_client = db.Historical()

In [3]:
cme = "GLBX.MDP3"
tz_chicago = ZoneInfo("America/Chicago")
now = datetime.datetime.now(tz=tz_chicago)
today = now.date()

In [4]:
yesterday = today - 2 * us_business_day

In [5]:
def dollarize_stats_at_settlement_price(
    stats: pd.DataFrame, instrument_defs: pd.DataFrame, approximate_margin: float
) -> pd.DataFrame:
    required_defs = ["unit_of_measure_qty"]
    if all(required in stats.columns for required in required_defs):
        extended_stats = stats.copy()
    else:
        extended_stats = stats.reset_index().merge(
            instrument_defs[["symbol"] + required_defs],
            left_on="Symbol",
            right_on="symbol",
        )
    extended_stats["Cleared volume($)"] = (
        extended_stats["unit_of_measure_qty"]
        * extended_stats["Settlement price"]
        * extended_stats["Cleared volume"]
    )
    extended_stats["Cleared volume/Margin"] = (
        extended_stats["unit_of_measure_qty"]
        * extended_stats["Settlement price"]
        / approximate_margin
    )
    extended_stats["Margin Volume($)"] = (
        approximate_margin * extended_stats["Cleared volume"]
    )
    extended_stats["Open interest($)"] = (
        extended_stats["unit_of_measure_qty"]
        * extended_stats["Settlement price"]
        * extended_stats["Open interest"]
    )
    cols = list(stats.columns)
    new_columns = [
        "unit_of_measure_qty",
        "Cleared volume($)",
        "Cleared volume/Margin",
        "Margin Volume($)",
        "Open interest($)",
    ]
    cols.extend(new_columns)
    extended_stats = extended_stats.set_index(["Trade date", "Symbol"])[cols]
    return extended_stats

In [6]:
cache_scope = "futures_intro"


@disk_cache(scope=cache_scope)
def cached_definitions(
    dataset: str,
    symbols: Iterable[str],
    date: datetime.date,
    client: db.Historical,
) -> pd.DataFrame:
    return client.timeseries.get_range(
        dataset=dataset,
        symbols=symbols,
        start=date,
        schema="definition",
    ).to_df()


@disk_cache(scope=cache_scope)
def cached_stats(
    dataset: str,
    symbols: Iterable[str],
    start: pd.Timestamp | datetime.date,
    client: db.Historical,
    end: pd.Timestamp | datetime.date | None = None,
) -> pd.DataFrame:
    return client.timeseries.get_range(
        dataset=dataset,
        schema="statistics",
        symbols=symbols,
        start=start,
        end=end,
    ).to_df()


start_of_this_year = datetime.date(2026, 1, 1)
symbols = (
    "ESH6",
    "ESM6",
    "ESU6",
)
raw_stats = cached_stats(
    dataset=cme,
    symbols=symbols,
    start=start_of_this_year,
    end=yesterday,
    client=futures_client,
)

In [7]:
instrument_defs = cached_definitions(
    dataset=cme,
    symbols=symbols,
    date=start_of_this_year,
    client=futures_client,
)

In [8]:
some_def_cols = [
    "instrument_id",
    "raw_symbol",
    "expiration",
    "unit_of_measure",
    "unit_of_measure_qty",
    "min_price_increment",
    "currency",
    "group",
    "exchange",
    "security_type",
    "high_limit_price",
    "low_limit_price",
]

In [9]:
stats = get_official_stats(raw_stats, instrument_defs)
instrument_defs[some_def_cols]

,instrument_id,raw_symbol,expiration,unit_of_measure,unit_of_measure_qty,min_price_increment,currency,group,exchange,security_type,high_limit_price,low_limit_price
ts_recv,,,,,,,,,,,,
2026-01-01 00:00:00+00:00,42140870,ESU6,2026-09-18 13:30:00+00:00,IPNT,50.0,0.25,USD,ES,XCME,FUT,7565.50,6595.50
2026-01-01 00:00:00+00:00,42140878,ESH6,2026-03-20 13:30:00+00:00,IPNT,50.0,0.25,USD,ES,XCME,FUT,7464.25,6494.25
2026-01-01 00:00:00+00:00,42140864,ESM6,2026-06-18 13:30:00+00:00,IPNT,50.0,0.25,USD,ES,XCME,FUT,7516.00,6546.00


## Nominal Value of Cleared Volume

Futures prices are often converted to _nominal values_ to express the dollar value
of the market exposure. That is, the prices are multiplied by the contract size.

The daily trading volume can be converted to dollar value
using the nominal value of the settlement price.

In [10]:
plot_df = stats.reset_index()
plot_df["Symbol"].unique()
plot_df = stats.reset_index()
plot_groups = plot_df.groupby("Symbol")
colors = {group: next(color_palette) for group in plot_groups.groups}

x_col = "Trade date"
plot_cols = ["Cleared volume", "Open interest"]
subplot_titles = plot_cols

fig = make_subplots(
    rows=len(plot_cols),
    cols=1,
    shared_xaxes="all",
    subplot_titles=subplot_titles,
)

for symbol in symbols:
    df = plot_groups.get_group(symbol)
    fig.add_trace(
        go.Scatter(
            x=df[x_col],
            y=df[plot_cols[0]],
            name=symbol,
            line=dict(color=colors[symbol]),
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=df[x_col],
            y=df[plot_cols[1]],
            name=symbol,
            line=dict(color=colors[symbol]),
            showlegend=False,
        ),
        row=2,
        col=1,
    )


fig.update_layout(
    height=600,
    width=600,
    title_text="Cleared volume and open interest over time for ES 2025 contracts",
)
fig.show()

## Comparing Nominal S&P Mini Cleared Volume to ETF Trade Values

Trade values for futures and ETFs can be compared using the nominal
futures value and the value of the traded volume in the ETF,
but the different accounting procedures for these products means
the comparison is not exactly apple to apples.

In [11]:
etfs = ("SPY", "VOO", "IVV")

In [12]:
us_equity_summary = "EQUS.SUMMARY"
equity_cache_dir = Path("~/.data/finm37000_equity").expanduser()
equity_def_date = datetime.date(2025, 8, 15)
with temp_env(FINM37000_CACHE_DIR=str(equity_cache_dir)):
    etf_defs = cached_definitions(
        dataset=us_equity_summary,
        symbols=etfs,
        date=equity_def_date,
        client=equity_client,
    )

WARNING:
If you have a pay-per-use Databento plan that allows you to access equity data,
the equity summary `"statistics"` schema
is very different from futures `"statistics"` schema.
It includes last trade data for all trades across exchanges
(Nasdaq Last Sale Plus, NLS+)
which Databento consolidates into their `ohclv-1d` schema.
So getting ETF volume from the stats table would
be similar to gathering futures volume
by downloading all the trades.
We use OHLCV data instead.

In [13]:
@disk_cache(scope=cache_scope)
def cached_cost(
    dataset: str,
    symbols: Iterable[str],
    start: pd.Timestamp | datetime.date,
    end: pd.Timestamp,
    client: db.Historical,
) -> pd.DataFrame:
    cost = client.metadata.get_cost(
        dataset=dataset,
        schema="statistics",
        symbols=symbols,
        start=start,
        end=end,
    )
    df = pd.DataFrame(
        [
            {
                "start": start,
                "end": end,
                "cost": cost,
            }
        ]
    )
    return df


with temp_env(FINM37000_CACHE_DIR=str(equity_cache_dir)):
    etf_volume_cost = cached_cost(
        dataset=us_equity_summary,
        symbols=etfs,
        start=start_of_this_year,
        end=yesterday,
        client=equity_client,
    )
print(f"${round(etf_volume_cost, 2)}")

$        start        end   cost
0  2026-01-01 2026-09-10  132.2


/var/folders/8y/2f_yzjcd2tz12vswl92n4vpw0000gp/T/ipykernel_15124/2484046929.py:36: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(f"${round(etf_volume_cost, 2)}")


In [14]:
@disk_cache(scope=cache_scope)
def cached_daily_ohlcv(
    dataset: str,
    symbols: Iterable[str],
    start: pd.Timestamp | datetime.date,
    end: pd.Timestamp,
    client: db.Historical,
) -> pd.DataFrame:
    return client.timeseries.get_range(
        dataset=dataset,
        schema="ohlcv-1d",
        symbols=symbols,
        start=start,
        end=end,
    ).to_df()


with temp_env(FINM37000_CACHE_DIR=str(equity_cache_dir)):
    etf_raw_ohlcv = cached_daily_ohlcv(
        dataset=us_equity_summary,
        symbols=etfs,
        start=start_of_this_year,
        end=yesterday,
        client=equity_client,
    )

In [15]:
etf_raw_ohlcv

,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol
ts_event,,,,,,,,,
2026-01-02 00:00:00+00:00,35,90,17039,630.635,631.66,625.210,628.30,11337024,VOO
2026-01-02 00:00:00+00:00,35,90,15144,685.710,686.87,679.820,683.17,89377203,SPY
2026-01-02 00:00:00+00:00,35,90,8863,688.740,689.90,682.880,686.19,7056867,IVV
2026-01-05 00:00:00+00:00,35,90,8863,689.590,692.47,689.420,690.79,11227873,IVV
2026-01-05 00:00:00+00:00,35,90,17039,631.380,634.04,631.230,632.46,16175621,VOO
...,...,...,...,...,...,...,...,...,...
2026-09-08 00:00:00+00:00,35,90,15144,769.070,769.70,765.140,765.96,44746095,SPY
2026-09-08 00:00:00+00:00,35,90,8863,772.760,773.32,768.540,769.20,16249848,IVV
2026-09-09 00:00:00+00:00,35,90,8863,767.710,768.14,764.632,766.10,8162185,IVV


In [16]:
def dollarize_etf_volume(
    stats: pd.DataFrame,
    defs: pd.DataFrame,
) -> pd.DataFrame:
    extended_stats = stats.reset_index()
    extended_stats["Trade date"] = extended_stats["ts_event"].dt.date
    extended_stats = extended_stats.merge(defs, on="instrument_id").reset_index()
    extended_stats["Symbol"] = extended_stats["raw_symbol"]
    extended_stats["Volume($)"] = extended_stats["close"] * extended_stats["volume"]
    cols = ["volume", "Volume($)", "close", "unit_of_measure_qty"]
    extended_stats = extended_stats.set_index(["Trade date", "Symbol"])[cols]
    return extended_stats


etfs_dollarized = dollarize_etf_volume(etf_raw_ohlcv, etf_defs)

In [17]:
etf_defs

,ts_event,rtype,publisher_id,instrument_id,raw_symbol,security_update_action,instrument_class,min_price_increment,display_factor,expiration,...,sub_fraction,underlying_product,maturity_month,maturity_day,maturity_week,user_defined_instrument,contract_multiplier_unit,flow_schedule_type,tick_rule,symbol
ts_recv,,,,,,,,,,,,,,,,,,,,,
2025-08-15 07:05:04.745241953+00:00,2025-08-15 07:05:04.745050570+00:00,19,90,8863,IVV,A,K,NaN,100000.0,NaT,...,255,255,255,255,255,N,127,127,255,IVV
2025-08-15 07:05:04.793858246+00:00,2025-08-15 07:05:04.793663666+00:00,19,90,15144,SPY,A,K,NaN,100000.0,NaT,...,255,255,255,255,255,N,127,127,255,SPY
2025-08-15 07:05:04.807830231+00:00,2025-08-15 07:05:04.807635899+00:00,19,90,17039,VOO,A,K,NaN,100000.0,NaT,...,255,255,255,255,255,N,127,127,255,VOO


In [18]:
plot_futures = dollarize_stats_at_settlement_price(
    stats,
    instrument_defs,
    approximate_margin=25_000,
).reset_index()
futures_groups = plot_futures.groupby("Symbol")
plot_etfs = etfs_dollarized.reset_index()
etf_groups = plot_etfs.groupby("Symbol")

x_col = "Trade date"
plot_cols = ["Cleared volume($)", "Volume($)"]
subplot_titles = plot_cols

fig = make_subplots(
    rows=len(plot_cols),
    cols=1,
    shared_xaxes="all",
    shared_yaxes="all",
    subplot_titles=subplot_titles,
)

for symbol in symbols:
    df = futures_groups.get_group(symbol)
    fig.add_trace(
        go.Scatter(
            x=df[x_col],
            y=df[plot_cols[0]],
            name=symbol,
        ),
        row=1,
        col=1,
    )
for etf in etfs:
    df = etf_groups.get_group(etf)
    fig.add_trace(
        go.Scatter(
            x=df[x_col],
            y=df[plot_cols[1]],
            name=etf,
        ),
        row=2,
        col=1,
    )


fig.update_layout(
    height=600,
    width=600,
    title_text="Cleared volume in ES vs. ETFs",
)
fig.show()

## Comparing S&P Mini Nominal Value/Margin to ETF Trade Values

We have seen that futures accounting is fundamentally different from stock accounting.
In particular, the settlement value of the cleared volume represents
a leveraged position with only the fractional margin backing the position.

The amount of leveraged exposure can be used instead. Here I plot the nominal
cleared value per margin, assuming a constant S&P Mini margin of $25,000. There
can be leveraged positions in the ETF market as well,
which I am not approximating or accounting for here.

By this metric, the nominal value per invested value is much smaller
in the (highly leveraged) futures market.

In [19]:
plot_cols = ["Cleared volume/Margin", "Volume($)"]
subplot_titles = ["Cleared volume/Margin", "Volume($)"]

fig = make_subplots(
    rows=len(plot_cols),
    cols=1,
    shared_xaxes="all",
    shared_yaxes="all",
    subplot_titles=subplot_titles,
)

for symbol in symbols:
    df = futures_groups.get_group(symbol)
    fig.add_trace(
        go.Scatter(
            x=df[x_col],
            y=df[plot_cols[0]],
            name=symbol,
        ),
        row=1,
        col=1,
    )
for etf in etfs:
    df = etf_groups.get_group(etf)
    fig.add_trace(
        go.Scatter(
            x=df[x_col],
            y=df[plot_cols[1]],
            name=etf,
        ),
        row=2,
        col=1,
    )


fig.update_layout(
    height=600,
    width=600,
    title_text="Cleared volume per margin in ES vs. ETFs",
)
fig.show()

## Comparing S&P Mini Margin Investment to some ETF Trade Value

Yet another way to compare these markets is to look at the dollar
amount invested. For futures, approximate this with a constant margin
value of $25,000. For ETFs, approximate this as the trade volume assuming
no special leveraging arrangements.

This metric shows a comparison of dollar value invested but not the scale
of potential returns.

In [20]:
plot_cols = ["Margin Volume($)", "Volume($)"]
subplot_titles = plot_cols

fig = make_subplots(
    rows=len(plot_cols),
    cols=1,
    shared_xaxes="all",
    shared_yaxes="all",
    subplot_titles=subplot_titles,
)

for symbol in symbols:
    df = futures_groups.get_group(symbol)
    fig.add_trace(
        go.Scatter(
            x=df[x_col],
            y=df[plot_cols[0]],
            name=symbol,
        ),
        row=1,
        col=1,
    )
for etf in etfs:
    df = etf_groups.get_group(etf)
    fig.add_trace(
        go.Scatter(
            x=df[x_col],
            y=df[plot_cols[1]],
            name=etf,
        ),
        row=2,
        col=1,
    )


fig.update_layout(
    height=600,
    width=600,
    title_text="Cleared volume per margin in ES vs. ETFs",
)
fig.show()

Which market reflects more trading volume in dollar terms?